# 05 - Bayesian networks: prediction, diagnosis and FTA-to-BN reasoning

## Learning objectives

- Apply Bayes theorem numerically
- Distinguish prior, likelihood and posterior
- Compare predictive and diagnostic inference
- Update TK-101 risk using sensor evidence


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. Why Bayes?

FTA usually asks a forward question: given basic-event probabilities, how likely is the top event? Operation often requires the inverse question: **after observing evidence, how likely is a fault?** This is diagnostic inference.


## 2. Bayes theorem by hand

$$P(F|E)=\frac{P(E|F)P(F)}{P(E|F)P(F)+P(E|\neg F)P(\neg F)}.$$

Use $P(F)=0.02$, detector sensitivity 0.90 and false-positive rate 0.05.


In [ ]:
prior=.02; sens=.90; fp=.05
posterior=sens*prior/(sens*prior+fp*(1-prior))
print(f"Prior={prior:.3f}; posterior={posterior:.3f}")


The posterior is approximately 27%. A positive indication substantially increases the fault probability but does not establish certainty. The base rate matters.


In [ ]:
priors=np.linspace(.001,.2,100)
posts=sens*priors/(sens*priors+fp*(1-priors))
plt.plot(priors,posts); plt.plot(priors,priors,'--'); plt.xlabel('prior'); plt.ylabel('posterior'); plt.grid(alpha=.3); plt.show()


## 3. Predictive versus diagnostic direction

Predictive: $P(E|F)$ asks what evidence to expect given a fault. Diagnostic: $P(F|E)$ asks how likely a fault is given evidence. These are generally different quantities.


## 4. A small Bayesian network by enumeration

`SensorFault -> BadMeasurement -> MissedIntervention`. Factor the joint distribution as $P(F,M,I)=P(F)P(M|F)P(I|M)$.


In [ ]:
P_F=.02
P_M_given_F={False:.03,True:.90}
P_I_given_M={False:.04,True:.70}
def joint(f,m,i):
    pf=P_F if f else 1-P_F
    pm=P_M_given_F[f] if m else 1-P_M_given_F[f]
    pi=P_I_given_M[m] if i else 1-P_I_given_M[m]
    return pf*pm*pi
rows=[]
for f in [False,True]:
 for m in [False,True]:
  for i in [False,True]: rows.append({'F':f,'M':m,'I':i,'P':joint(f,m,i)})
jt=pd.DataFrame(rows); print('sum=',jt.P.sum()); jt


In [ ]:
subset=jt[jt.M]
print('P(F=True | M=True)=',subset[subset.F].P.sum()/subset.P.sum())


## 5. FTA-to-BN reasoning

FTA gates can be represented by deterministic or near-deterministic conditional probability tables. A BN allows evidence at an intermediate variable and diagnostic inference. Shared events and dependencies need explicit treatment when translating a fault tree.


## 6. TK-101: posterior sensor-fault probability -> updated overflow risk

In the fault tree, `MANUAL_FAIL = LT101_LOW OR OPERATOR_MISS`. Replacing the static `P(LT101_LOW)=0.02` by a diagnostic posterior makes the top-event probability conditional on evidence, subject to the model assumptions.


In [ ]:
p_ch=.08; p_auto=1-(1-.01)*(1-.005); p_op=.04
def overflow(p_sensor):
    p_manual=1-(1-p_sensor)*(1-p_op)
    return p_ch*p_auto*p_manual
print('prior risk    ',overflow(prior))
print('evidence risk ',overflow(posterior))


## 7. Calibration

Sensitivity, false-positive rate and prior probability need evidence. Sources can include proof tests, historical data, expert elicitation or validated diagnostic experiments. The posterior is only as credible as these inputs.


In [ ]:
from safetycourse.bayes import BinaryNode,BinaryBayesNet
nodes=[BinaryNode('Fault',[],{():.02}),BinaryNode('Evidence',['Fault'],{(False,):.05,(True,):.90})]
bn=BinaryBayesNet(nodes)
print(bn.query('Fault',{'Evidence':True}))


## Try it!

1. Reduce sensitivity to 0.70.
2. Increase the false-positive rate to 0.15.
3. Compute P(F|M=False) by enumeration.
4. Add MaintenanceOverdue as a parent influencing the fault prior and label this as a modeling assumption.


## Summary

A Bayesian network supports evidence-updated diagnostic beliefs. The next unit introduces **time and state transitions** through Markov models and a Dynamic HAZOP illustration.
